In [ ]:
import random
import joblib
from nn_models   import *
from nn_data     import *
from nn_training import *
from torch.utils.data import DataLoader
from sklearn.preprocessing import StandardScaler

In [ ]:
DATA_PATH = '/path'
BEST_HP_PATH = '/path'

### 1. Load data

In [ ]:
features = KEY_FEATURES + CONTROL_FEATURES
df_train = pd.read_csv(data_path + 'train.csv')
df_test  = pd.read_csv(data_path + 'test.csv')
df_test = df_test.dropna(subset='pm25')

X_train = np.array(df_train[features])
y_train = np.array(df_train['pm25_filled'])
X_test = np.array(df_test[features])
y_test = np.array(df_test['pm25'])

### 2. Standardize data

In [ ]:
loaded_study = joblib.load(BEST_HP_PATH)
best         = loaded_study.best_params
best_trial   = loaded_study.best_trial
best_epoch   = best_trial.user_attrs["best_epoch"]

In [4]:
# Fit statistic on train only
feature_scaler = StandardScaler()
target_scaler  = StandardScaler()

feature_scaler.fit(X_train)
target_scaler.fit(y_train.reshape(-1, 1))

# Transform the data
train_dataset = MLPDataset(X_train, y_train, feature_scaler, target_scaler)
test_dataset  = MLPDataset(X_test,  y_test,  feature_scaler, target_scaler)

train_loader = DataLoader(train_dataset, batch_size=best['batch_size'], shuffle=True)

### 3. Train the model

In [5]:
physical_zero_std = -feature_scaler.mean_[:N_SOURCES] / feature_scaler.scale_[:N_SOURCES]

In [ ]:
seed_everything()

model = TwoStageMLP(
                    n_sources   = N_SOURCES,            
                    n_fkey      = N_FKEY,
                    n_fcontrol  = N_FCONTROL,       
                    h1          = best_trial.user_attrs['h1'], 
                    h2          = best_trial.user_attrs['h2'], 
                    hout        = best_trial.user_attrs['hout'],     
                    hh1         = best_trial.user_attrs['hh1'], 
                    hh2         = best_trial.user_attrs['hh2'], 
                    drop_rate   = best['drop_rate'], 
                    hdrop_rate  = best['hdrop_rate'], 
                    hsk         = best['hsk'],          
                    threshold   = physical_zero_std,
                    **source_sum_scaler_params(feature_scaler, target_scaler, N_SOURCES)) 

optimizer = torch.optim.Adam(model.parameters(), lr=best['lr'])

epochs = best_epoch
train_losses = train_TwostageMLP(model, train_loader, optimizer, epochs, lambda_sim=best['lambda_sim'])

### 4. Collect predictions

In [ ]:
df_test_pred   = collect_TwoStageMLP_preds(test_dataset.X, 
                                           df_test,  
                                           model, 
                                           target_scaler, 
                                           feature_scaler, 
                                           CMAQ_FEATURES, 
                                           len(features))